# 07 — Limpieza bibliográfica

Esta libreta realiza **exclusivamente** la limpieza de los campos bibliográficos de
`autores_unam_normalizados.csv`.

La unidad de trazabilidad es:

`Base_origen + indice`

Se conserva exactamente una fila por **PUBLICACIÓN + INVESTIGADOR UNAM**.

### No se hace en esta fase

- No se deduplican publicaciones.
- No se fusionan fuentes.
- No se hace record linkage.
- No se investigan autores ni afiliaciones.
- No se modifica `Autor_norm`, `Afiliacion1`, `Afiliacion2`, `Area` ni `SubArea`.
- `SubArea` debe permanecer vacía.
- No se generan datos bibliográficos a partir de aproximaciones.

### Prioridad de recuperación

1. Valor válido ya presente en el registro actual.
2. `Canonico_Union_Trabajo.csv` por `Base_origen + indice`.
3. `UNAM_Completo_Corregido.xlsx` **solo mediante DOI exacto**, cuando el valor es inequívoco y además pasa la validación del campo.
4. Si no hay evidencia segura, se deja vacío o se registra para revisión.


In [1]:
import os
import re
import html
import unicodedata
from collections import Counter

import pandas as pd

# -------------------------------------------------------------------
# Rutas simples
# -------------------------------------------------------------------
# La libreta se ejecuta desde notebooks/.
ARCHIVO_ENTRADA = "../04_Limpieza/02_normalizacion/autores_unam_normalizados.csv"

# Estas dos rutas siguen la estructura usada previamente en el repositorio.
# Si tus archivos de referencia están en otra carpeta, cambia SOLO estas líneas.
ARCHIVO_CANONICO = "../02_modelo_canonico/03_union/Canonico_Union_Trabajo.csv"
ARCHIVO_CONTROL_UNAM = "../00_control/UNAM_Completo_Corregido.csv"

CARPETA_SALIDA = "../04_Limpieza/03_limpieza_bibliografica"
ARCHIVO_SALIDA = CARPETA_SALIDA + "/autores_unam_limpios.csv"
ARCHIVO_REVISION = CARPETA_SALIDA + "/casos_revision_bibliografica.csv"

os.makedirs(CARPETA_SALIDA, exist_ok=True)

COLUMNAS_CANONICAS = [
    "Base_origen",
    "Fuente_origen",
    "indice",
    "Titulo",
    "Año",
    "Autor_norm",
    "Afiliacion1",
    "Afiliacion2",
    "ISBN",
    "ISSN",
    "Doi",
    "URL",
    "Area",
    "SubArea",
    "Keywords",
    "Abstract",
]

CLAVE_PUBLICACION = ["Base_origen", "indice"]

COLUMNAS_NO_MODIFICAR = [
    "Base_origen",
    "Fuente_origen",
    "indice",
    "Autor_norm",
    "Afiliacion1",
    "Afiliacion2",
    "Area",
    "SubArea",
]

COLUMNAS_LIMPIAR = [
    "Titulo",
    "Año",
    "ISBN",
    "ISSN",
    "Doi",
    "URL",
    "Keywords",
    "Abstract",
]

COLUMNAS_IGUALES_POR_PUBLICACION = [
    "Titulo",
    "Año",
    "ISBN",
    "ISSN",
    "Doi",
    "URL",
    "Area",
    "SubArea",
    "Keywords",
    "Abstract",
]

print("Entrada:", ARCHIVO_ENTRADA)
print("Referencia canónica:", ARCHIVO_CANONICO)
print("Control UNAM:", ARCHIVO_CONTROL_UNAM)
print("Salida:", ARCHIVO_SALIDA)


Entrada: ../04_Limpieza/02_normalizacion/autores_unam_normalizados.csv
Referencia canónica: ../02_modelo_canonico/03_union/Canonico_Union_Trabajo.csv
Control UNAM: ../00_control/UNAM_Completo_Corregido.csv
Salida: ../04_Limpieza/03_limpieza_bibliografica/autores_unam_limpios.csv


In [2]:
# -------------------------------------------------------------------
# Lectura: todo como texto para no volver a deformar identificadores
# -------------------------------------------------------------------
df_raw = pd.read_csv(
    ARCHIVO_ENTRADA,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
)

canonico = pd.read_csv(
    ARCHIVO_CANONICO,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
)

control_unam = pd.read_csv(
    ARCHIVO_CONTROL_UNAM,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig",
)

filas_antes = len(df_raw)
columnas_fisicas_antes = len(df_raw.columns)

faltantes = [c for c in COLUMNAS_CANONICAS if c not in df_raw.columns]
if faltantes:
    raise ValueError("Faltan columnas canónicas en la entrada: " + str(faltantes))

columnas_extra = [c for c in df_raw.columns if c not in COLUMNAS_CANONICAS]

# No se eliminan silenciosamente columnas extra que contengan información.
extras_con_datos = []
for c in columnas_extra:
    if df_raw[c].astype(str).str.strip().ne("").any():
        extras_con_datos.append(c)

if extras_con_datos:
    raise ValueError(
        "Hay columnas físicas extra con contenido. Revisar antes de continuar: "
        + str(extras_con_datos)
    )

# Las columnas extra vacías (por ejemplo Unnamed: 16...) sí se descartan.
df = df_raw[COLUMNAS_CANONICAS].copy()

# Guardamos una copia exacta de las 16 columnas para comprobar al final
# que autores, afiliaciones, área, índice y trazabilidad no cambiaron.
df_original = df.copy()

# SubArea debe llegar vacía y permanecer vacía.
if df["SubArea"].astype(str).str.strip().ne("").any():
    raise ValueError(
        "SubArea contiene valores. Esta fase no debe vaciar ni corregir "
        "silenciosamente una SubArea previamente poblada."
    )

print("Filas:", filas_antes)
print("Columnas físicas de entrada:", columnas_fisicas_antes)
print("Columnas extra vacías:", columnas_extra)
print("Columnas de trabajo:", len(df.columns))


Filas: 4266
Columnas físicas de entrada: 20
Columnas extra vacías: ['Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19']
Columnas de trabajo: 16


## Funciones deterministas

Las funciones siguientes evitan reglas agresivas:

- no se usa `.title()` para títulos;
- no se reconstruyen ISBN desde notación científica;
- ISBN e ISSN se validan como identificadores;
- DOI no se obtiene del título;
- una URL vacía no se crea automáticamente desde DOI;
- en `Keywords` solo se eliminan URLs/fragmentos web inequívocos;
- las etiquetas HTML de formato se eliminan conservando el contenido;
- en `Abstract` no se elimina copyright ni se reescribe el texto;
- no se usa una expresión genérica `<...>` para HTML porque podría borrar desigualdades matemáticas legítimas.


In [3]:
# -------------------------------------------------------------------
# Expresiones regulares
# -------------------------------------------------------------------
RE_CIENTIFICA = re.compile(
    r"(?i)^[+-]?(?:\d+(?:\.\d*)?|\.\d+)[Ee][+-]?\d+$"
)

RE_ENTIDAD_HTML = re.compile(
    r"&(?:[A-Za-z][A-Za-z0-9]+|#\d+|#x[0-9A-Fa-f]+);"
)

# Solo etiquetas de FORMATO conocidas.
# No usamos <[^>]+> porque en abstracts aparecen expresiones matemáticas
# como n/2 < t < n que deben conservarse.
RE_TAG_FORMATO = re.compile(
    r"(?i)</?\s*(?:sup|sub|i|b|em|strong|span|p|div|br)\b[^>]*>"
)

RE_PARAMETRO_WEB = re.compile(
    r"(?i)\b(?:arnumber|isnumber|tag|refinements|querytext|partnerid|md5|eid)\s*="
)

RE_URL_KEYWORD = re.compile(
    r"(?i)(?:https?://|www\.|ieeexplore\.ieee\.org|scopus\.com)"
)

# En Keywords la URL termina en un separador de lista.
RE_URL_COMPLETA_KEYWORD = re.compile(r"(?i)https?://[^;\s,|]+")
RE_WWW_KEYWORD = re.compile(r"(?i)\bwww\.[^;\s,|]+")


def texto(valor):
    """Convierte None/NaN a cadena vacía sin introducir 'nan'."""
    if valor is None:
        return ""
    s = str(valor)
    if s.lower() == "nan":
        return ""
    return s


def decodificar_html_repetido(valor, max_iteraciones=4):
    """Decodifica &amp;, &quot;, &#...; y dobles codificaciones controladamente."""
    s = texto(valor)
    for _ in range(max_iteraciones):
        nuevo = html.unescape(s)
        if nuevo == s:
            break
        s = nuevo
    return s


def nfc(valor):
    return unicodedata.normalize("NFC", texto(valor))


def clave_publicacion(base, indice):
    """Normalización SOLO para construir la clave; no modifica las columnas de salida."""
    return (texto(base).strip(), texto(indice).strip())


def limpiar_titulo(valor):
    s = decodificar_html_repetido(valor)
    s = unicodedata.normalize("NFC", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalizar_anio(valor):
    """
    Acepta YYYY o YYYY.0 y devuelve YYYY.
    No extrae años desde textos más largos.
    """
    s = texto(valor).strip()
    m = re.fullmatch(r"(\d{4})(?:\.0+)?", s)
    return m.group(1) if m else ""


def isbn10_valido(s):
    s = s.upper()
    if not re.fullmatch(r"\d{9}[\dX]", s):
        return False

    total = 0
    for i, ch in enumerate(s):
        numero = 10 if ch == "X" else int(ch)
        total += (10 - i) * numero

    return total % 11 == 0


def isbn13_valido(s):
    if not re.fullmatch(r"\d{13}", s):
        return False

    total = 0
    for i, ch in enumerate(s[:12]):
        total += int(ch) * (1 if i % 2 == 0 else 3)

    digito = (10 - (total % 10)) % 10
    return digito == int(s[-1])


def analizar_isbn(valor):
    """
    Devuelve:
      - ISBN válidos sin puntuación;
      - fragmentos no interpretables;
      - indicador de notación científica.

    Una cadena científica nunca se expande ni redondea.
    """
    s = nfc(decodificar_html_repetido(valor)).strip()

    if not s:
        return [], [], False

    s = re.sub(r"(?i)\bISBN(?:-1[03])?\b\s*:?", "", s)

    hay_cientifica = bool(
        re.search(
            r"(?i)(?<!\w)[+-]?(?:\d+(?:\.\d*)?|\.\d+)[Ee][+-]?\d+(?!\w)",
            s,
        )
    )

    # ISBN no utiliza '/', por eso este separador permite separar
    # contaminaciones tipo /2025/07 sin reconstruir ningún dígito.
    partes = re.split(r"\s*(?:;|,|\||/)\s*", s)

    validos = []
    invalidos = []

    for parte in partes:
        parte = parte.strip(" \t\r\n.:")
        if not parte:
            continue

        if RE_CIENTIFICA.fullmatch(parte):
            invalidos.append(parte)
            continue

        chars = re.sub(r"[^0-9Xx]", "", parte).upper()

        if len(chars) == 13 and isbn13_valido(chars):
            validos.append(chars)
        elif len(chars) == 10 and isbn10_valido(chars):
            validos.append(chars)
        else:
            invalidos.append(parte)

    # Orden original, sin duplicados.
    validos = list(dict.fromkeys(validos))

    return validos, invalidos, hay_cientifica


def isbn_normalizado_si_seguro(valor):
    validos, invalidos, hay_cientifica = analizar_isbn(valor)

    # En los datos reales aparecen residuos de fecha después de un ISBN válido,
    # por ejemplo /2025/07. El ISBN se acepta únicamente si su checksum es válido.
    residuos_serios = []
    for frag in invalidos:
        frag_limpio = re.sub(r"\D", "", frag)
        if not re.fullmatch(r"\d{2}|\d{4}", frag_limpio):
            residuos_serios.append(frag)

    if validos and not hay_cientifica and not residuos_serios:
        return "; ".join(validos), True

    return "", False


def issn_checksum_valido(chars):
    chars = chars.upper()

    if not re.fullmatch(r"\d{7}[\dX]", chars):
        return False

    valores = [int(ch) for ch in chars[:7]]
    ultimo = 10 if chars[-1] == "X" else int(chars[-1])

    total = sum(v * peso for v, peso in zip(valores, range(8, 1, -1)))
    total += ultimo

    return total % 11 == 0


def analizar_issn(valor):
    s = nfc(decodificar_html_repetido(valor))

    if not s.strip():
        return [], []

    s = re.sub(r"(?i)\b(?:e-?issn|issn)\b\s*:?", "", s)
    partes = re.split(r"\s*(?:;|,|\||/)\s*", s)

    validos = []
    invalidos = []

    for parte in partes:
        parte = parte.strip(" \t\r\n.:")
        if not parte:
            continue

        chars = re.sub(r"[^0-9Xx]", "", parte).upper()

        if len(chars) == 8 and issn_checksum_valido(chars):
            validos.append(chars[:4] + "-" + chars[4:])
        else:
            invalidos.append(parte)

    validos = list(dict.fromkeys(validos))
    return validos, invalidos


def issn_normalizado_si_seguro(valor):
    validos, invalidos = analizar_issn(valor)

    if validos and not invalidos:
        return "; ".join(validos), True

    return "", False


def normalizar_doi(valor):
    s = nfc(decodificar_html_repetido(valor)).strip()

    if not s:
        return ""

    s = re.sub(
        r"(?i)^\s*(?:https?://(?:dx\.)?doi\.org/|doi\s*:\s*)",
        "",
        s,
    ).strip()

    if re.search(r"\s", s):
        return ""

    s = s.lower()

    if not re.fullmatch(r"10\.\d{4,9}/\S+", s):
        return ""

    return s


def normalizar_url(valor):
    s = nfc(decodificar_html_repetido(valor)).strip()

    if not s:
        return ""

    if re.fullmatch(r"(?i)https?://\S+", s):
        return s

    return ""


def url_es_resolvedor_doi(url):
    return bool(
        re.match(r"(?i)^https?://(?:dx\.)?doi\.org/", texto(url).strip())
    )


def keywords_tienen_basura_web(valor):
    s = nfc(decodificar_html_repetido(valor))
    return bool(RE_URL_KEYWORD.search(s) or RE_PARAMETRO_WEB.search(s))


def limpiar_keywords(valor):
    s = nfc(decodificar_html_repetido(valor))

    if not s.strip():
        return ""

    # Etiquetas de formato: conservar el contenido científico.
    s = re.sub(r"(?i)<\s*br\s*/?\s*>", ";", s)
    s = re.sub(r"(?i)</?\s*(?:p|div)\b[^>]*>", ";", s)
    s = re.sub(
        r"(?i)</?\s*(?:sup|sub|i|b|em|strong|span)\b[^>]*>",
        "",
        s,
    )

    # Quitar URL incrustada sin comerse las keywords que siguen.
    s = RE_URL_COMPLETA_KEYWORD.sub("", s)
    s = RE_WWW_KEYWORD.sub("", s)

    # Los datos reales utilizan sobre todo ; y , como separadores.
    partes = re.split(r"\s*(?:;|,|\r?\n|\||•)\s*", s)

    resultado = []
    vistos = set()

    for parte in partes:
        parte = re.sub(r"\s+", " ", parte).strip()

        if not parte:
            continue

        # Fragmentos inequívocos de una consulta o URL.
        if RE_PARAMETRO_WEB.search(parte) or RE_URL_KEYWORD.search(parte):
            continue

        # Duplicado EXACTO después de limpieza técnica.
        # No se hace fuzzy matching ni se colapsan variantes de mayúsculas.
        if parte not in vistos:
            resultado.append(parte)
            vistos.add(parte)

    return "; ".join(resultado)


def limpiar_abstract(valor):
    s = nfc(decodificar_html_repetido(valor))

    if not s.strip():
        return ""

    # Solo etiquetas HTML de formato conocidas.
    # Se conservan desigualdades, símbolos científicos y copyright.
    s = re.sub(r"(?i)<\s*br\s*/?\s*>", " ", s)
    s = re.sub(r"(?i)</?\s*(?:p|div)\b[^>]*>", " ", s)
    s = re.sub(
        r"(?i)</?\s*(?:sup|sub|i|b|em|strong|span)\b[^>]*>",
        "",
        s,
    )

    return s.strip()


def isbn_esta_en_notacion_cientifica(valor):
    return bool(
        re.search(
            r"(?i)(?<!\w)[+-]?(?:\d+(?:\.\d*)?|\.\d+)[Ee][+-]?\d+(?!\w)",
            texto(valor),
        )
    )


def isbn_tiene_problema(valor):
    s = texto(valor).strip()
    if not s:
        return False

    validos, invalidos, cientifica = analizar_isbn(s)
    return cientifica or not validos or bool(invalidos)


def issn_esta_en_formato_salida(valor):
    s = texto(valor).strip()
    if not s:
        return True

    partes = s.split("; ")
    return all(re.fullmatch(r"\d{4}-\d{3}[\dX]", p) for p in partes)


def tiene_html_reconocido(valor):
    s = texto(valor)
    return bool(RE_ENTIDAD_HTML.search(s) or RE_TAG_FORMATO.search(s))


In [4]:
# -------------------------------------------------------------------
# Diagnóstico inicial SOBRE LA ENTRADA, antes de limpiar
# -------------------------------------------------------------------
def contar_filas_publicaciones(mask):
    sub = df.loc[mask, CLAVE_PUBLICACION].copy()

    if len(sub) == 0:
        return 0, 0

    claves = {
        clave_publicacion(base, indice)
        for base, indice in zip(sub["Base_origen"], sub["indice"])
    }

    return len(sub), len(claves)


anio_punto_cero = df["Año"].str.strip().str.match(r"^\d{4}\.0+$")
anio_vacio = df["Año"].str.strip().eq("")

isbn_cientifico = df["ISBN"].apply(isbn_esta_en_notacion_cientifica)
isbn_invalido = df["ISBN"].apply(isbn_tiene_problema)

issn_no_formato = (
    df["ISSN"].str.strip().ne("")
    & ~df["ISSN"].apply(issn_esta_en_formato_salida)
)

doi_vacio = df["Doi"].str.strip().eq("")

url_vacia = df["URL"].str.strip().eq("")
url_invalida = (
    df["URL"].str.strip().ne("")
    & df["URL"].apply(lambda x: normalizar_url(x) == "")
)

keywords_basura = df["Keywords"].apply(keywords_tienen_basura_web)
keywords_vacias = df["Keywords"].str.strip().eq("")

abstract_vacio = df["Abstract"].str.strip().eq("")

html_por_campo = {}
for campo in ["Titulo", "URL", "Keywords", "Abstract"]:
    mask = df[campo].apply(tiene_html_reconocido)
    html_por_campo[campo] = contar_filas_publicaciones(mask)

# Prefijos DOI observados.
def tipo_prefijo_doi(valor):
    s = texto(valor).strip()

    if not s:
        return "VACIO"
    if re.match(r"(?i)^https://doi\.org/", s):
        return "https://doi.org/"
    if re.match(r"(?i)^http://doi\.org/", s):
        return "http://doi.org/"
    if re.match(r"(?i)^https?://dx\.doi\.org/", s):
        return "dx.doi.org/"
    if re.match(r"(?i)^doi\s*:", s):
        return "doi:"
    if s.startswith("10."):
        return "DOI puro"
    return "OTRO"


prefijos_doi = Counter(
    df.loc[df["Doi"].str.strip().ne(""), "Doi"].apply(tipo_prefijo_doi)
)

# Inconsistencias dentro de Base_origen + indice.
nunique_grupos = (
    df.groupby(CLAVE_PUBLICACION, sort=False)[COLUMNAS_IGUALES_POR_PUBLICACION]
    .nunique(dropna=False)
)

inconsistencias_por_campo = {
    campo: int((nunique_grupos[campo] > 1).sum())
    for campo in COLUMNAS_IGUALES_POR_PUBLICACION
}

grupos_inconsistentes = int((nunique_grupos > 1).any(axis=1).sum())

print("========== DIAGNÓSTICO INICIAL ==========")
print("Filas:", filas_antes)
print("Número de columnas físicas:", columnas_fisicas_antes)
print("Columnas extra:", columnas_extra)
print("Año con .0 (filas, publicaciones):", contar_filas_publicaciones(anio_punto_cero))
print("Año vacío (filas, publicaciones):", contar_filas_publicaciones(anio_vacio))
print("ISBN en notación científica:", contar_filas_publicaciones(isbn_cientifico))
print("ISBN con formato/contenido inválido:", contar_filas_publicaciones(isbn_invalido))
print("ISSN sin formato canónico:", contar_filas_publicaciones(issn_no_formato))
print("Prefijos DOI:", dict(prefijos_doi))
print("DOI vacíos:", contar_filas_publicaciones(doi_vacio))
print("URL vacías:", contar_filas_publicaciones(url_vacia))
print("URL inválidas no vacías:", contar_filas_publicaciones(url_invalida))
print("Keywords con URL/basura web:", contar_filas_publicaciones(keywords_basura))
print("Keywords vacías:", contar_filas_publicaciones(keywords_vacias))
print("Abstract vacíos:", contar_filas_publicaciones(abstract_vacio))
print("HTML/entidades por campo:", html_por_campo)
print("Grupos con inconsistencias bibliográficas:", grupos_inconsistentes)
print("Inconsistencias por campo:", inconsistencias_por_campo)

# Para los archivos recibidos este valor es 0.
# Si en otra ejecución aparece una inconsistencia previa, no se elige
# silenciosamente entre valores contradictorios.
if grupos_inconsistentes > 0:
    raise ValueError(
        "Existen publicaciones con metadatos bibliográficos contradictorios "
        "antes de la limpieza. Revisar esos grupos antes de ejecutar esta fase."
    )


========== DIAGNÓSTICO INICIAL ==========
Filas: 4266
Número de columnas físicas: 20
Columnas extra: ['Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19']
Año con .0 (filas, publicaciones): (0, 0)
Año vacío (filas, publicaciones): (272, 123)
ISBN en notación científica: (131, 63)
ISBN con formato/contenido inválido: (173, 105)
ISSN sin formato canónico: (2546, 1129)
Prefijos DOI: {'DOI puro': 4036}
DOI vacíos: (230, 71)
URL vacías: (584, 251)
URL inválidas no vacías: (0, 0)
Keywords con URL/basura web: (0, 0)
Keywords vacías: (18, 14)
Abstract vacíos: (2, 2)
HTML/entidades por campo: {'Titulo': (0, 0), 'URL': (0, 0), 'Keywords': (12, 4), 'Abstract': (2, 2)}
Grupos con inconsistencias bibliográficas: 0
Inconsistencias por campo: {'Titulo': 0, 'Año': 0, 'ISBN': 0, 'ISSN': 0, 'Doi': 0, 'URL': 0, 'Area': 0, 'SubArea': 0, 'Keywords': 0, 'Abstract': 0}


## Preparar las fuentes de referencia

`Canonico_Union_Trabajo.csv` se utiliza únicamente por la clave exacta
`Base_origen + indice`.

`UNAM_Completo_Corregido.xlsx` **no** tiene esa clave. Para evitar coincidencias aproximadas,
solo se usa como tercer nivel de evidencia cuando el DOI limpio coincide exactamente.
Además, el valor recuperado debe pasar la validación específica del campo.

Esto evita copiar ciegamente errores del archivo de control.


In [5]:
# -------------------------------------------------------------------
# Preparar Canonico_Union_Trabajo
# -------------------------------------------------------------------
faltantes_canonico = [
    c for c in COLUMNAS_CANONICAS
    if c not in canonico.columns
]

if faltantes_canonico:
    raise ValueError(
        "Faltan columnas en Canonico_Union_Trabajo: "
        + str(faltantes_canonico)
    )

# Clave temporal. No modifica indice en la salida.
canonico["_clave"] = [
    clave_publicacion(base, indice)
    for base, indice in zip(canonico["Base_origen"], canonico["indice"])
]

if canonico["_clave"].duplicated().any():
    duplicadas = canonico.loc[
        canonico["_clave"].duplicated(keep=False),
        ["Base_origen", "indice"],
    ]
    raise ValueError(
        "Canonico_Union_Trabajo no es único por Base_origen + indice.\n"
        + duplicadas.head(20).to_string(index=False)
    )

mapa_canonico = {
    row["_clave"]: row.to_dict()
    for _, row in canonico.iterrows()
}

claves_entrada = {
    clave_publicacion(base, indice)
    for base, indice in zip(df["Base_origen"], df["indice"])
}

faltan_en_canonico = sorted(claves_entrada - set(mapa_canonico))

if faltan_en_canonico:
    raise ValueError(
        "Hay publicaciones de la entrada sin referencia por Base_origen + indice: "
        + str(faltan_en_canonico[:20])
    )

print("Publicaciones de entrada:", len(claves_entrada))
print("Publicaciones cubiertas por Canonico_Union_Trabajo:", len(claves_entrada))
print("Filas de Canonico_Union_Trabajo:", len(canonico))

# -------------------------------------------------------------------
# Preparar UNAM_Completo_Corregido solo por DOI exacto
# -------------------------------------------------------------------
columnas_control_necesarias = [
    "Titulo", "Año", "ISBN", "ISSN", "Doi", "URL", "Keywords", "Abstract"
]

faltantes_control = [
    c for c in columnas_control_necesarias
    if c not in control_unam.columns
]

if faltantes_control:
    raise ValueError(
        "Faltan columnas en UNAM_Completo_Corregido: "
        + str(faltantes_control)
    )

control_unam["_Doi_limpio"] = control_unam["Doi"].apply(normalizar_doi)

control_por_doi = {}

for doi, grupo in control_unam.loc[
    control_unam["_Doi_limpio"].ne("")
].groupby("_Doi_limpio", sort=False):

    control_por_doi[doi] = grupo.copy()


def valor_unico_control(doi, campo):
    """
    Devuelve un valor no vacío únicamente cuando TODAS las filas del DOI
    permiten identificar un solo valor no vacío para ese campo.
    """
    if not doi or doi not in control_por_doi:
        return ""

    valores = []

    for valor in control_por_doi[doi][campo]:
        s = texto(valor).strip()
        if s and s not in valores:
            valores.append(s)

    return valores[0] if len(valores) == 1 else ""


dois_entrada = {
    normalizar_doi(v)
    for v in df["Doi"]
    if normalizar_doi(v)
}

coincidencias_doi_control = len(
    [doi for doi in dois_entrada if doi in control_por_doi]
)

print("DOI distintos de la entrada:", len(dois_entrada))
print("DOI con coincidencia exacta en el control UNAM:", coincidencias_doi_control)


Publicaciones de entrada: 2109
Publicaciones cubiertas por Canonico_Union_Trabajo: 2109
Filas de Canonico_Union_Trabajo: 2153
DOI distintos de la entrada: 436
DOI con coincidencia exacta en el control UNAM: 21


## Resolver una vez cada publicación

Como el archivo contiene varias filas por publicación, la limpieza se decide una sola vez por
`Base_origen + indice` y después se propaga a todos sus investigadores UNAM.

La lista de títulos para `AREA_SOSPECHOSA` procede exclusivamente de las observaciones de la
tutora. Se utiliza solo para **registrar** casos; `Area` nunca se modifica aquí.


In [6]:
# -------------------------------------------------------------------
# Títulos señalados por la tutora como ejemplos que corresponden a IA
# -------------------------------------------------------------------
# La hoja escaneada muestra algunos títulos truncados con "...".
# Se usan únicamente inicios de título suficientemente distintivos
# para generar una BANDERA DE REVISIÓN; nunca para modificar Area.
PREFIJOS_AREA_IA = [
    "Enhancing Perceptron Learning Through Bayesian Optimisation",
    "Smart Parking",
    "Neuroevolution-based multiobjective algorithm",
    "AI-Based Mobile App for Segmentation",
    "Multimodal Early Birth Weight Prediction Using Multiple Kernel Learning",
    "Machine Learning and Deep Learning Sentiment Analysis Models",
    "Unsupervised Anomaly Detection",
    "Characterization of wildfire severity using deep learning",
    "Comparative Analysis of Machine Learning and Transformer Models",
    "A Bayesian Machine Learning Model for Predicting",
    "AlzyFinder",
    "Causal Behaviour Modelling",
]

casos_revision = []
bibliografia_por_clave = {}
recuperaciones = Counter()

for (base, indice), grupo in df.groupby(CLAVE_PUBLICACION, sort=False):

    clave = clave_publicacion(base, indice)

    # El diagnóstico anterior comprobó que no hay contradicciones dentro del grupo.
    actual = grupo.iloc[0].to_dict()
    referencia = mapa_canonico[clave]

    # ---------------------------------------------------------------
    # DOI
    # ---------------------------------------------------------------
    doi_actual = normalizar_doi(actual["Doi"])
    doi_ref = normalizar_doi(referencia["Doi"])

    doi = doi_actual or doi_ref

    if not doi_actual and doi_ref:
        recuperaciones["DOI_desde_canonico"] += 1

    if actual["Doi"].strip() and not doi:
        casos_revision.append({
            "Base_origen": base,
            "indice": indice,
            "Titulo": limpiar_titulo(actual["Titulo"]),
            "Campo": "Doi",
            "Valor_actual": actual["Doi"],
            "Problema": "DOI_INVALIDO",
            "Accion_recomendada":
                "Revisar contra una fuente bibliográfica; no se confirmó un DOI válido.",
        })

    # DOI confirmado que permite consultar el archivo de control.
    doi_para_control = doi

    # ---------------------------------------------------------------
    # Título
    # ---------------------------------------------------------------
    titulo_actual = limpiar_titulo(actual["Titulo"])
    titulo_ref = limpiar_titulo(referencia["Titulo"])

    if (not titulo_actual or "�" in titulo_actual) and titulo_ref and "�" not in titulo_ref:
        titulo = titulo_ref
        recuperaciones["Titulo_desde_canonico"] += 1
    else:
        titulo = titulo_actual

    if (not titulo or "�" in titulo) and doi_para_control:
        valor_control = valor_unico_control(doi_para_control, "Titulo")
        titulo_control = limpiar_titulo(valor_control)

        if titulo_control and "�" not in titulo_control:
            titulo = titulo_control
            recuperaciones["Titulo_desde_control"] += 1

    # ---------------------------------------------------------------
    # Año
    # ---------------------------------------------------------------
    anio_actual = normalizar_anio(actual["Año"])
    anio_ref = normalizar_anio(referencia["Año"])

    if anio_actual:
        anio = anio_actual
    elif anio_ref:
        anio = anio_ref
        recuperaciones["Año_desde_canonico"] += 1
    else:
        anio_control = normalizar_anio(
            valor_unico_control(doi_para_control, "Año")
        )
        anio = anio_control

        if anio_control:
            recuperaciones["Año_desde_control"] += 1

    # ---------------------------------------------------------------
    # ISBN
    # ---------------------------------------------------------------
    isbn_actual, isbn_actual_ok = isbn_normalizado_si_seguro(actual["ISBN"])
    isbn_ref, isbn_ref_ok = isbn_normalizado_si_seguro(referencia["ISBN"])

    if isbn_actual_ok:
        isbn = isbn_actual
    elif isbn_ref_ok:
        isbn = isbn_ref
        recuperaciones["ISBN_desde_canonico"] += 1
    else:
        isbn_control, isbn_control_ok = isbn_normalizado_si_seguro(
            valor_unico_control(doi_para_control, "ISBN")
        )

        if isbn_control_ok:
            isbn = isbn_control
            recuperaciones["ISBN_desde_control"] += 1
        else:
            isbn = ""

            if actual["ISBN"].strip() or referencia["ISBN"].strip():
                casos_revision.append({
                    "Base_origen": base,
                    "indice": indice,
                    "Titulo": titulo,
                    "Campo": "ISBN",
                    "Valor_actual": actual["ISBN"],
                    "Problema": "ISBN_NO_RECUPERABLE",
                    "Accion_recomendada":
                        "No se recuperó un ISBN exacto y válido; conservar vacío.",
                })

    # ---------------------------------------------------------------
    # ISSN
    # ---------------------------------------------------------------
    issn_actual, issn_actual_ok = issn_normalizado_si_seguro(actual["ISSN"])
    issn_ref, issn_ref_ok = issn_normalizado_si_seguro(referencia["ISSN"])

    if issn_actual_ok:
        issn = issn_actual
    elif issn_ref_ok:
        issn = issn_ref
        recuperaciones["ISSN_desde_canonico"] += 1
    else:
        issn_control, issn_control_ok = issn_normalizado_si_seguro(
            valor_unico_control(doi_para_control, "ISSN")
        )

        if issn_control_ok:
            issn = issn_control
            recuperaciones["ISSN_desde_control"] += 1
        else:
            issn = ""

            if actual["ISSN"].strip() or referencia["ISSN"].strip():
                casos_revision.append({
                    "Base_origen": base,
                    "indice": indice,
                    "Titulo": titulo,
                    "Campo": "ISSN",
                    "Valor_actual": actual["ISSN"],
                    "Problema": "ISSN_INVALIDO",
                    "Accion_recomendada":
                        "No se recuperó un ISSN válido; conservar vacío.",
                })

    # ---------------------------------------------------------------
    # URL
    # ---------------------------------------------------------------
    url_actual = normalizar_url(actual["URL"])
    url_ref = normalizar_url(referencia["URL"])

    if url_actual:
        url = url_actual
    elif url_ref:
        url = url_ref
        recuperaciones["URL_desde_canonico"] += 1
    else:
        url_control = normalizar_url(
            valor_unico_control(doi_para_control, "URL")
        )

        # Regla explícita: no llenar una URL vacía con un resolvedor DOI.
        if url_control and not url_es_resolvedor_doi(url_control):
            url = url_control
            recuperaciones["URL_desde_control"] += 1
        else:
            url = ""

            if actual["URL"].strip() or referencia["URL"].strip():
                casos_revision.append({
                    "Base_origen": base,
                    "indice": indice,
                    "Titulo": titulo,
                    "Campo": "URL",
                    "Valor_actual": actual["URL"],
                    "Problema": "URL_INVALIDA",
                    "Accion_recomendada":
                        "No se recuperó una URL bibliográfica válida; conservar vacío.",
                })

    # ---------------------------------------------------------------
    # Keywords
    # ---------------------------------------------------------------
    kw_actual = actual["Keywords"]
    kw_ref = referencia["Keywords"]

    if kw_actual.strip() and not keywords_tienen_basura_web(kw_actual):
        kw_fuente = kw_actual

    elif kw_ref.strip() and not keywords_tienen_basura_web(kw_ref):
        kw_fuente = kw_ref
        recuperaciones["Keywords_desde_canonico"] += 1

    else:
        kw_control = valor_unico_control(doi_para_control, "Keywords")

        if kw_control.strip() and not keywords_tienen_basura_web(kw_control):
            kw_fuente = kw_control
            recuperaciones["Keywords_desde_control"] += 1
        else:
            # Si todas las fuentes están contaminadas, se intenta conservar
            # únicamente términos que sobrevivan la limpieza determinista.
            kw_fuente = kw_actual or kw_ref

    keywords = limpiar_keywords(kw_fuente)

    if (kw_actual.strip() or kw_ref.strip()) and not keywords:
        casos_revision.append({
            "Base_origen": base,
            "indice": indice,
            "Titulo": titulo,
            "Campo": "Keywords",
            "Valor_actual": kw_actual,
            "Problema": "KEYWORDS_CONTAMINADAS",
            "Accion_recomendada":
                "No quedaron keywords bibliográficas seguras; conservar vacío y revisar.",
        })

    # ---------------------------------------------------------------
    # Abstract
    # ---------------------------------------------------------------
    abstract_actual = actual["Abstract"]
    abstract_ref = referencia["Abstract"]

    if abstract_actual.strip() and "�" not in abstract_actual:
        abstract_fuente = abstract_actual

    elif abstract_ref.strip() and "�" not in abstract_ref:
        abstract_fuente = abstract_ref
        recuperaciones["Abstract_desde_canonico"] += 1

    else:
        abstract_control = valor_unico_control(doi_para_control, "Abstract")

        if abstract_control.strip() and "�" not in abstract_control:
            abstract_fuente = abstract_control
            recuperaciones["Abstract_desde_control"] += 1
        else:
            abstract_fuente = abstract_actual or abstract_ref

    abstract = limpiar_abstract(abstract_fuente)

    if (abstract_actual.strip() or abstract_ref.strip()) and not abstract:
        casos_revision.append({
            "Base_origen": base,
            "indice": indice,
            "Titulo": titulo,
            "Campo": "Abstract",
            "Valor_actual": abstract_actual,
            "Problema": "ABSTRACT_PROBLEMATICO",
            "Accion_recomendada":
                "El abstract no pudo conservarse de manera segura; revisar manualmente.",
        })

    # ---------------------------------------------------------------
    # Area: SOLO bandera. Nunca modificar.
    # ---------------------------------------------------------------
    prefijo_area = next(
        (
            p for p in PREFIJOS_AREA_IA
            if titulo.casefold().startswith(p.casefold())
        ),
        None,
    )

    if prefijo_area and actual["Area"].strip().upper() != "IA":
        casos_revision.append({
            "Base_origen": base,
            "indice": indice,
            "Titulo": titulo,
            "Campo": "Area",
            "Valor_actual": actual["Area"],
            "Problema": "AREA_SOSPECHOSA",
            "Accion_recomendada":
                "La tutora señaló este título entre ejemplos que corresponden a IA; "
                "no se modifica Area en esta fase.",
        })

    bibliografia_por_clave[clave] = {
        "Titulo": titulo,
        "Año": anio,
        "ISBN": isbn,
        "ISSN": issn,
        "Doi": doi,
        "URL": url,
        "Keywords": keywords,
        "Abstract": abstract,
    }

print("Publicaciones resueltas:", len(bibliografia_por_clave))
print("Recuperaciones por fuente:", dict(recuperaciones))
print("Casos de revisión:", len(casos_revision))


Publicaciones resueltas: 2109
Recuperaciones por fuente: {'ISBN_desde_canonico': 63, 'ISSN_desde_canonico': 258, 'Año_desde_control': 4, 'ISSN_desde_control': 3, 'URL_desde_control': 2}
Casos de revisión: 10


In [7]:
# -------------------------------------------------------------------
# Propagar la misma bibliografía a todas las filas de cada publicación
# -------------------------------------------------------------------
df_final = df.copy()

for campo in COLUMNAS_LIMPIAR:
    df_final[campo] = [
        bibliografia_por_clave[
            clave_publicacion(base, indice)
        ][campo]
        for base, indice in zip(
            df_final["Base_origen"],
            df_final["indice"],
        )
    ]

# Area y SubArea NO se tocan.
# El orden de filas tampoco se modifica.
df_final = df_final[COLUMNAS_CANONICAS]

print("Filas antes:", len(df_original))
print("Filas después:", len(df_final))
print("Columnas finales:", list(df_final.columns))


Filas antes: 4266
Filas después: 4266
Columnas finales: ['Base_origen', 'Fuente_origen', 'indice', 'Titulo', 'Año', 'Autor_norm', 'Afiliacion1', 'Afiliacion2', 'ISBN', 'ISSN', 'Doi', 'URL', 'Area', 'SubArea', 'Keywords', 'Abstract']


## Validaciones obligatorias

Esta sección detiene la ejecución si se viola alguna regla estructural o si se modifica
una columna protegida.


In [8]:
# -------------------------------------------------------------------
# 1-12. Estructura, filas y columnas protegidas
# -------------------------------------------------------------------
assert filas_antes == 4266, (
    "La entrada recibida para esta fase debe tener 4,266 filas. "
    f"Se encontraron {filas_antes}."
)

assert len(df_final) == filas_antes
assert len(df_final.columns) == 16
assert list(df_final.columns) == COLUMNAS_CANONICAS

for campo in COLUMNAS_NO_MODIFICAR:
    assert df_final[campo].equals(df_original[campo]), (
        f"La columna protegida {campo} cambió."
    )

assert df_final["SubArea"].str.strip().eq("").all()
assert not any(c.startswith("Unnamed") for c in df_final.columns)

# -------------------------------------------------------------------
# 13. Año = vacío o cuatro dígitos
# -------------------------------------------------------------------
assert df_final["Año"].apply(
    lambda x: x == "" or bool(re.fullmatch(r"\d{4}", x))
).all()

# -------------------------------------------------------------------
# 14. ISBN: nunca notación científica y, si existe, ISBN válido
# -------------------------------------------------------------------
for valor in df_final["ISBN"]:
    if not valor:
        continue

    assert not isbn_esta_en_notacion_cientifica(valor)

    for isbn in valor.split("; "):
        assert (
            (len(isbn) == 13 and isbn13_valido(isbn))
            or
            (len(isbn) == 10 and isbn10_valido(isbn))
        )

# -------------------------------------------------------------------
# 15. ISSN en NNNN-NNNN, X final permitida
# -------------------------------------------------------------------
for valor in df_final["ISSN"]:
    if not valor:
        continue

    for issn in valor.split("; "):
        assert re.fullmatch(r"\d{4}-\d{3}[\dX]", issn)
        assert issn_checksum_valido(issn.replace("-", ""))

# -------------------------------------------------------------------
# 16-17. DOI puro
# -------------------------------------------------------------------
assert df_final["Doi"].apply(
    lambda x: x == "" or x.startswith("10.")
).all()

assert ~df_final["Doi"].str.contains(
    "doi.org",
    case=False,
    regex=False,
).any()

assert ~df_final["Doi"].str.contains(r"\s", regex=True).any()

# -------------------------------------------------------------------
# 18-19. Keywords sin URL ni parámetros web
# -------------------------------------------------------------------
assert ~df_final["Keywords"].apply(
    lambda x: bool(RE_URL_KEYWORD.search(x))
).any()

assert ~df_final["Keywords"].apply(
    lambda x: bool(RE_PARAMETRO_WEB.search(x))
).any()

# -------------------------------------------------------------------
# 20. No columnas Unnamed
# -------------------------------------------------------------------
assert not any(c.startswith("Unnamed") for c in df_final.columns)

# -------------------------------------------------------------------
# 21. Igualdad bibliográfica dentro de cada publicación
# -------------------------------------------------------------------
nunique_final = (
    df_final.groupby(CLAVE_PUBLICACION, sort=False)[
        COLUMNAS_IGUALES_POR_PUBLICACION
    ]
    .nunique(dropna=False)
)

assert not (nunique_final > 1).any().any()

# -------------------------------------------------------------------
# 22. Unicode NFC en los campos textuales limpiados
# -------------------------------------------------------------------
for campo in ["Titulo", "Keywords", "Abstract"]:
    assert df_final[campo].apply(
        lambda x: unicodedata.normalize("NFC", x) == x
    ).all()

# -------------------------------------------------------------------
# Solo columnas autorizadas cambiaron
# -------------------------------------------------------------------
cambios_por_columna = {}

for campo in COLUMNAS_CANONICAS:
    cambios_por_columna[campo] = int(
        (df_final[campo] != df_original[campo]).sum()
    )

columnas_que_cambiaron = {
    campo
    for campo, n in cambios_por_columna.items()
    if n > 0
}

solo_bibliograficas_autorizadas = columnas_que_cambiaron.issubset(
    set(COLUMNAS_LIMPIAR)
)

assert solo_bibliograficas_autorizadas

print("VALIDACIONES COMPLETADAS CORRECTAMENTE")
print("Cambios por columna:", cambios_por_columna)
print(
    "Solo columnas bibliográficas autorizadas cambiaron:",
    solo_bibliograficas_autorizadas,
)


VALIDACIONES COMPLETADAS CORRECTAMENTE
Cambios por columna: {'Base_origen': 0, 'Fuente_origen': 0, 'indice': 0, 'Titulo': 20, 'Año': 9, 'Autor_norm': 0, 'Afiliacion1': 0, 'Afiliacion2': 0, 'ISBN': 1978, 'ISSN': 2549, 'Doi': 1414, 'URL': 2, 'Area': 0, 'SubArea': 0, 'Keywords': 1751, 'Abstract': 49}
Solo columnas bibliográficas autorizadas cambiaron: True


In [9]:
# -------------------------------------------------------------------
# Guardar salida final y, solo si existe, el archivo de revisión
# -------------------------------------------------------------------
df_final.to_csv(
    ARCHIVO_SALIDA,
    index=False,
    encoding="utf-8-sig",
)

columnas_revision = [
    "Base_origen",
    "indice",
    "Titulo",
    "Campo",
    "Valor_actual",
    "Problema",
    "Accion_recomendada",
]

df_revision = pd.DataFrame(
    casos_revision,
    columns=columnas_revision,
)

# Una sola fila por anomalía de publicación/campo/problema.
if len(df_revision) > 0:
    df_revision = df_revision.drop_duplicates(
        subset=[
            "Base_origen",
            "indice",
            "Campo",
            "Problema",
        ],
        keep="first",
    )

    df_revision.to_csv(
        ARCHIVO_REVISION,
        index=False,
        encoding="utf-8-sig",
    )

else:
    # Evita dejar un archivo viejo de revisión de una ejecución anterior.
    if os.path.exists(ARCHIVO_REVISION):
        os.remove(ARCHIVO_REVISION)

print("Archivo final:", ARCHIVO_SALIDA)

if len(df_revision) > 0:
    print("Archivo de revisión:", ARCHIVO_REVISION)
else:
    print("No fue necesario generar casos_revision_bibliografica.csv")


Archivo final: ../04_Limpieza/03_limpieza_bibliografica/autores_unam_limpios.csv
Archivo de revisión: ../04_Limpieza/03_limpieza_bibliografica/casos_revision_bibliografica.csv


In [10]:
# -------------------------------------------------------------------
# Resumen breve solicitado
# -------------------------------------------------------------------
print()
print("========== RESUMEN FINAL ==========")
print("Filas antes:", len(df_original))
print("Filas después:", len(df_final))
print("Años corregidos:", cambios_por_columna["Año"])
print("ISBN recuperados/corregidos:", cambios_por_columna["ISBN"])
print("ISSN corregidos:", cambios_por_columna["ISSN"])
print("DOI normalizados:", cambios_por_columna["Doi"])
print("Keywords limpiadas:", cambios_por_columna["Keywords"])
print("Abstract corregidos:", cambios_por_columna["Abstract"])
print("Casos pendientes:", len(df_revision))
print(
    "Solo columnas bibliográficas autorizadas cambiaron:",
    solo_bibliograficas_autorizadas,
)



========== RESUMEN FINAL ==========
Filas antes: 4266
Filas después: 4266
Años corregidos: 9
ISBN recuperados/corregidos: 1978
ISSN corregidos: 2549
DOI normalizados: 1414
Keywords limpiadas: 1751
Abstract corregidos: 49
Casos pendientes: 10
Solo columnas bibliográficas autorizadas cambiaron: True
